In [ ]:
from typing import List

import numpy as np
import pandas as pd
from lightkurve import LightCurve
from matplotlib import pyplot as plt

from exo_finder.default_datasets import sunlike_lightcurves_ds, exo_dataset, candidate_dataset
from exo_finder.visualization.lightcurve_plotting import plot_lightcurve_ax

from paths import LC_STATS_RESULT_FILE

In [ ]:
exo_db = exo_dataset.load_known_exoplanets_dataset()
toi_db = candidate_dataset.load_candidate_exoplanets_dataset()
lc_db = sunlike_lightcurves_ds.load_lightcurve_dataset()
exoplanets_ids = set(exo_db.unique_tic_ids) | set(toi_db.unique_tic_ids)

In [ ]:
# Load datasets
combined_analysis = pd.read_feather(LC_STATS_RESULT_FILE)

# exclude invalid lightcurves
combined_analysis = combined_analysis.loc[
    (combined_analysis["normalized_std"] > 0) & (combined_analysis["split_count"] > 0)
]

# exclude lightcurves with known planets
combined_analysis = combined_analysis[~np.isin(combined_analysis["tic_id"], list(exoplanets_ids))]
combined_analysis

In [ ]:
COLUMNS_OF_INTEREST = ["median_f", "min_f", "max_f", "length", "gap_ratio", "normalized_std", "mad"]
combined_analysis[COLUMNS_OF_INTEREST].describe()

In [ ]:
def plot_histograms(df: pd.DataFrame, title: str, columns: list[str] = COLUMNS_OF_INTEREST):
    c = 2
    # Calculate the number of rows and columns for subplots
    num_cols = len(columns)
    num_rows = (num_cols + c - 1) // c

    # Create subplots
    fig, axs = plt.subplots(num_rows, c, figsize=(15, num_rows * 3))

    # Flatten axs if necessary
    if num_rows > 1:
        axs = axs.flatten()

    # Plot histograms for each column
    for i, col in enumerate(columns):
        # Remove outliers 3-sigma away from the mean
        median = np.median(df[col])
        std = df[col].std()
        filtered_data = df[(df[col] >= median - 3 * std) & (df[col] <= median + 3 * std)]

        ax = axs[i]
        ax.hist(filtered_data[col], bins=100, color="skyblue", edgecolor="black")
        ax.set_title(col)
        ax.set_xlabel("Value")
        ax.set_ylabel("Frequency")
        ax.grid(True)

    # Adjust layout
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


plot_histograms(
    combined_analysis,
    title="Parameters distributions for all dataset",
    columns=["gap_ratio", "mad", "normalized_std", "length"],
)

In [ ]:
def plot_observations(lcs: List[LightCurve], obs_ids: pd.Series, title: str):
    c = 2
    r = (len(lcs) + 1) // c

    fig, axes = plt.subplots(r, c, figsize=(15, 3 * r))
    axes = axes.flatten()

    for lc, obsid, ax in zip(lcs, obs_ids, axes):
        plot_lightcurve_ax(lc, ax, title=lc.meta["OBJECT"] + f" - obs_id: {obsid}")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


def get_observations(df: pd.DataFrame) -> List[LightCurve]:
    lcs = []
    for _, row in df.iterrows():
        lcs.append(lc_db.load_by_obs_id(row.obs_id))
    return lcs


def show_high_and_low_parametrer(df: pd.DataFrame, parameter: str):
    high_val = df.sort_values(parameter, ascending=False).head(4)
    low_val = df.sort_values(parameter, ascending=True).head(4)

    high_lc = get_observations(high_val)
    low_lc = get_observations(low_val)

    plot_observations(high_lc, high_val.obs_id, title=f"Highest {parameter}")
    plot_observations(low_lc, low_val.obs_id, title=f"Lowest {parameter}")


for col in COLUMNS_OF_INTEREST:
    show_high_and_low_parametrer(combined_analysis, col)

In [ ]:
def plot_min_max(parameters: list[str], df: pd.DataFrame):
    rows = len(parameters)
    fig, axes = plt.subplots(rows, 2, figsize=(15, rows * 3))
    for i, parameter in enumerate(parameters):
        ax1, ax2 = axes[i]
        sorted_df = df.sort_values(parameter, ascending=True)
        low_val = sorted_df.iloc[1]
        high_val = sorted_df.iloc[-2]

        low_lc = lc_db.load_by_obs_id(low_val.obs_id).remove_outliers()
        high_lc = lc_db.load_by_obs_id(high_val.obs_id).remove_outliers()

        plot_lightcurve_ax(low_lc, ax1, title=f"Lowest {parameter}")
        plot_lightcurve_ax(high_lc, ax2, title=f"Highest {parameter}")
    plt.tight_layout()
    plt.show()


plot_min_max(parameters=["gap_ratio", "mad", "normalized_std"], df=combined_analysis)

# Defines Rules for dataset selection
## Gap Ratio should be small, limit MAD and STD

In [ ]:
for r in [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]:
    print(f"Number of gap_ratio less than {r}: {(combined_analysis.gap_ratio <= r).sum()}")

In [ ]:
gap_ratio_threshold = 0.15
mad_threshold = np.percentile(combined_analysis["mad"], 70)
std_threshold = np.percentile(combined_analysis["normalized_std"], 70)

subset = combined_analysis[
    (combined_analysis.gap_ratio <= gap_ratio_threshold)
    & (combined_analysis.mad <= mad_threshold)
    & (combined_analysis.normalized_std <= std_threshold)
]

print("Number of lightcurves:", len(subset))
print("Number of individual stars:", len(subset["tic_id"].unique()))
plot_histograms(
    subset,
    title="Parameter distributions for filtered dataset",
    columns=["gap_ratio", "mad", "normalized_std", "length"],
)

In [ ]:
for col in COLUMNS_OF_INTEREST:
    show_high_and_low_parametrer(subset, col)